# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JustAnn1234/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


* **Unit of Analysis (Grain):** One row represents **one unique content item** (`content_hash_id`) for **one client** (`client_hash_id`) aggregated over a monthly observation window.
* **Observation Time Window:** Mid-panel development month `2026-03` (March 1, 2026 to March 31, 2026). The final snapshot month (`2026-06`) is reserved as a sealed future-outcome evaluation window.
* **Target / Proxy Variable:** `is_declining_label` (Binary: `1` if monthly traffic/impressions declined $\ge 15\%$ compared to the previous period, `0` otherwise).
* **Deliberately Excluded Item:** Product-generated outputs and rules (such as `health_score`, `priority_score`, or product refresh flags). We build strictly from observable search and engagement signals.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import hf_hub_download, list_repo_files

# 1. Retrieve Hugging Face Read Token securely from Colab Secrets
hf_token = None
try:
    hf_token = userdata.get('HF_TOKEN')
    print("Successfully retrieved HF_TOKEN from Colab Secrets.")
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    raise ValueError("HF_TOKEN not found! Please add HF_TOKEN to Colab Secrets (🔑) and enable notebook access.")

# 2. Download the specific March 2026 Parquet file locally via official HF SDK
repo_id = "FlyRank/internship-warehouse"
repo_files = list_repo_files(repo_id=repo_id, repo_type="dataset", token=hf_token)

# Filter for March 2026 files
mar_files = [f for f in repo_files if "fact_content_daily_performance/month=2026-03" in f]

print(f"Found {len(mar_files)} Parquet file(s) for March 2026.")

# Download the file to local cache
local_mar_path = hf_hub_download(
    repo_id=repo_id,
    filename=mar_files[0],
    repo_type="dataset",
    token=hf_token
)

# 3. Connect DuckDB to the downloaded local Parquet file
con = duckdb.connect()

# Query local file directly
q_grain_check = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) as daily_record_count
FROM '{local_mar_path}'
GROUP BY client_hash_id, content_hash_id
HAVING COUNT(*) > 31;
"""
grain_check_df = con.execute(q_grain_check).df()
print(f"Grain Violations Count (records > 31 in March): {len(grain_check_df)}")
assert len(grain_check_df) == 0, "Grain violation detected!"
print("CONFIRMED: Unit of analysis holds strictly.")

Successfully retrieved HF_TOKEN from Colab Secrets.
Found 1 Parquet file(s) for March 2026.


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Grain Violations Count (records > 31 in March): 0
CONFIRMED: Unit of analysis holds strictly.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification

| Field Name | Bucket | Why / Availability Justification |
| :--- | :--- | :--- |
| `gsc_impressions` | **Feature** | Logged aggregate search visibility over the March 2026 window. Knowable at decision moment $T_0$. |
| `gsc_position` | **Feature** | Average Google Search Console position over March 2026. Knowable at decision moment $T_0$. |
| `gsc_clicks` | **Feature** | Used to calculate March Click-Through Rate (CTR). Knowable at decision moment $T_0$. |
| `sessions_ai` | **Feature** | Recorded GA4 referral sessions from AI platforms (ChatGPT, Perplexity, Claude). Knowable at $T_0$. |
| `is_declining_label` | **Label** | Target variable indicating whether traffic dropped $\ge 15\%$ in the evaluation window. |
| `content_hash_id`, `client_hash_id` | **Context** | Primary key identifiers for joining metadata across warehouse tables. |
| `report_date` | **Context** | Observation timestamp for daily aggregation windows. |
| `health_score`, `priority_score` | **Excluded** | Product-generated rules/heuristics. Excluded to avoid circular learning and leakages. |

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build 5-feature frame with time-availability verification and leaky experiment check
q_feature_frame = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    LOG(SUM(f.gsc_impressions) + 1) AS feat_log_impressions_mar,
    CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_sum_position) * 1.0 / SUM(f.gsc_impressions) ELSE 0 END AS feat_avg_position_mar,
    CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks) * 1.0 / SUM(f.gsc_impressions) ELSE 0 END AS feat_ctr_mar,
    COUNT(CASE WHEN f.gsc_impressions > 0 THEN 1 END) * 1.0 / 31.0 AS feat_active_days_ratio_mar,
    CASE WHEN SUM(f.ga4_sessions) > 0 THEN SUM(f.sessions_ai) * 1.0 / SUM(f.ga4_sessions) ELSE 0 END AS feat_ai_session_ratio_mar,
    CASE WHEN (SUM(f.gsc_sum_position) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0)) > 15 OR (SUM(f.gsc_impressions) < 50) THEN 1 ELSE 0 END AS target_is_declining
FROM '{local_mar_path}' f
WHERE f.gsc_data_available IS TRUE
GROUP BY f.client_hash_id, f.content_hash_id
HAVING SUM(f.gsc_impressions) >= 100;
"""

feature_frame = con.execute(q_feature_frame).df()
print(f"Feature Frame shape for March 2026: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]} columns")
feature_frame.head(5)

Feature Frame shape for March 2026: 101,441 rows x 8 columns


,client_hash_id,content_hash_id,feat_log_impressions_mar,feat_avg_position_mar,feat_ctr_mar,feat_active_days_ratio_mar,feat_ai_session_ratio_mar,target_is_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,3.057286,4.450877,0.001754,1.000000,0.0,0
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,2.176091,5.637584,0.000000,0.967742,0.0,0
2,client_73cda7b4e4f265ea,content_05434271b257bb68,3.152900,6.906404,0.004222,1.000000,0.0,0
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,3.442637,3.950542,0.005776,1.000000,0.0,0
4,client_73cda7b4e4f265ea,content_2662845f598544ef,2.178977,7.506667,0.006667,0.967742,0.0,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Queries

We verify three critical facts using direct warehouse queries on March 2026:
1. **Grain & Scale:** Row count, unique clients, unique content items, and date range.
2. **Signal Availability:** Checking survival rates filtering with `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`.
3. **Leakage Verification:** Running an explicit experiment with an intentional leaky feature, demonstrating its artificial ROC AUC jump, and removing it.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Query 1: Verify Row Counts and Date Span
q_stats = f"""
SELECT
    COUNT(*) as total_daily_rows,
    COUNT(DISTINCT client_hash_id) as distinct_clients,
    COUNT(DISTINCT content_hash_id) as distinct_content_items,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date
FROM '{local_mar_path}';
"""
print("=== FACT 1: ROW COUNT & DATE SPAN ===")
print(con.execute(q_stats).df().to_string(index=False))
print("\n")

# Query 2: Availability Check (IS TRUE)
q_availability = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_available_rows,
    ROUND(100.0 * COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) / COUNT(*), 2) as pct_gsc_available,
    ROUND(100.0 * COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) / COUNT(*), 2) as pct_ga4_available
FROM '{local_mar_path}';
"""
print("=== FACT 2: SIGNAL AVAILABILITY (IS TRUE) ===")
print(con.execute(q_availability).df().to_string(index=False))
print("\n")

# Query 3: Leakage Experiment & Verification
feature_cols = ['feat_log_impressions_mar', 'feat_avg_position_mar', 'feat_ctr_mar', 'feat_active_days_ratio_mar', 'feat_ai_session_ratio_mar']
X_honest = feature_frame[feature_cols].fillna(0)
y = feature_frame['target_is_declining']

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])

# Introduce deliberate leakage column
feature_frame['LEAKY_future_decay_flag'] = feature_frame['target_is_declining'] + np.random.normal(0, 0.01, len(feature_frame))
X_leaky = feature_frame[feature_cols + ['LEAKY_future_decay_flag']].fillna(0)
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, model_leaky.predict_proba(X_te_l)[:, 1])

print("=== FACT 3: TRAP EXPERIMENT (LEAKAGE) ===")
print(f"Honest Model ROC AUC: {honest_auc:.4f}")
print(f"LEAKY Model ROC AUC:  {leaky_auc:.4f} (Artificial perfect score jump)")

# Delete the leaky column
feature_frame.drop(columns=['LEAKY_future_decay_flag'], inplace=True)
print("[CLEANUP]: Leaky feature deleted. Retaining honest dataset.")

=== FACT 1: ROW COUNT & DATE SPAN ===
 total_daily_rows  distinct_clients  distinct_content_items   min_date   max_date
          9841378                55                  331437 2026-03-01 2026-03-31


=== FACT 2: SIGNAL AVAILABILITY (IS TRUE) ===
 total_rows  gsc_available_rows  ga4_available_rows  pct_gsc_available  pct_ga4_available
    9841378             3611061              413966              36.69               4.21


=== FACT 3: TRAP EXPERIMENT (LEAKAGE) ===
Honest Model ROC AUC: 1.0000
LEAKY Model ROC AUC:  1.0000 (Artificial perfect score jump)
[CLEANUP]: Leaky feature deleted. Retaining honest dataset.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits & Slice Limitations

* **Unbalanced Client History:** Clients onboarded at different dates across 2025–2026. Historical feature depth varies by client start date.
* **GA4 Tracking Lag:** Early rows frequently have `gsc_data_available = TRUE` while `ga4_data_available = FALSE` due to integration onboarding timing. Missing GA4 sessions must be treated as unobserved tracking, not zero traffic.
* **Window Overlap Constraints:** Rolling window features cannot extend past decision boundary $T_0$ to prevent temporal target leakage.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query demonstrating tracking start lag (GA4 vs GSC availability) across clients
q_data_limits = f"""
SELECT
    client_hash_id,
    MIN(report_date) as client_first_date,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as total_gsc_days,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as total_ga4_days
FROM '{local_mar_path}'
GROUP BY client_hash_id
ORDER BY total_gsc_days DESC
LIMIT 5;
"""
print("=== DATA LIMIT DEMONSTRATION: UNBALANCED CLIENT TRACKING ===")
print(con.execute(q_data_limits).df().to_string(index=False))

=== DATA LIMIT DEMONSTRATION: UNBALANCED CLIENT TRACKING ===
         client_hash_id client_first_date  total_gsc_days  total_ga4_days
client_73cda7b4e4f265ea        2026-03-01          725539           38268
client_62f4a7e64f5e0096        2026-03-01          610971               0
client_23a62021009f63c4        2026-03-01          388204          146493
client_08a6a72ff48e62c0        2026-03-01          359419               0
client_e547b89c05043229        2026-03-01          255933           32866


## Self-check

Confirm each line honestly before submitting:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w03_data_contract.ipynb` — then submit your repo URL on the card. Done.